# Notebook 03 — Evaluation, Validation & Comparison

**Phase 3 of the Nokia Network Anomaly Detection PRD (Week 5)**

This notebook walks through the full Phase 3 evaluation:

| Section | PRD Reference |
|---------|---------------|
| Load trained models & test data | Phase 2 output |
| Score all models, tune threshold (FPR ≤ 15%) | NFR2.2, R3 |
| Precision / Recall / F1 / ROC-AUC per model | NFR2.1, §7.4.1 |
| ROC curves + Precision-Recall curves | §7.4.1 |
| Confusion matrices | §7.4.1 |
| Model comparison table | Phase 3 |
| Feature importance via SHAP | R7 |
| K-fold cross-validation (stability) | Phase 3 |
| Error analysis: FP / FN breakdown | Phase 3 |
| Export `results/metrics.json` | §11.3 |

In [ ]:
import os, sys
# Make src/ importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, auc,
)

MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# PRD targets
PRD_FPR_TARGET    = 0.15
PRD_PRECISION_MIN = 0.85
PRD_RECALL_MIN    = 0.80
PRD_AUC_MIN       = 0.90

print('Setup complete.')

## 1. Load Test Data & Models

In [ ]:
from models.isolation_forest import IsolationForestDetector
from models.lof import LOFDetector

X_test = joblib.load(os.path.join(MODELS_DIR, 'X_test.pkl'))
y_test_raw = joblib.load(os.path.join(MODELS_DIR, 'y_test.pkl'))
y_test = (pd.Series(y_test_raw).reset_index(drop=True) != 0).astype(int)

iforest = IsolationForestDetector.load()
lof     = LOFDetector.load()

print(f'Test rows : {len(X_test)}')
print(f'Anomalies : {int(y_test.sum())}  ({100*y_test.mean():.1f}%)')
print(f'Features  : {X_test.shape[1]}')

## 2. Threshold Tuning — FPR ≤ 15% (PRD NFR2.2, R3)

In [ ]:
def tune_threshold(y_true, y_score, max_fpr=PRD_FPR_TARGET):
    fpr_arr, tpr_arr, thr_arr = roc_curve(y_true, y_score)
    valid = np.where(fpr_arr <= max_fpr)[0]
    idx = valid[-1] if len(valid) else 0
    idx = min(idx, len(thr_arr) - 1)
    return float(thr_arr[idx]), float(fpr_arr[idx]), float(tpr_arr[idx])

X_np = X_test.values
if_score  = iforest.anomaly_score(X_np)
lof_score = lof.anomaly_score(X_np)

if_thr,  if_fpr_t,  if_tpr_t  = tune_threshold(y_test, if_score)
lof_thr, lof_fpr_t, lof_tpr_t = tune_threshold(y_test, lof_score)

iforest.set_threshold(if_thr)
lof.set_threshold(lof_thr)

if_pred  = iforest.predict(X_np)
lof_pred = lof.predict(X_np)

print(f'IF  tuned threshold = {if_thr:.4f}  → FPR={if_fpr_t:.3f}  TPR={if_tpr_t:.3f}')
print(f'LOF tuned threshold = {lof_thr:.4f}  → FPR={lof_fpr_t:.3f}  TPR={lof_tpr_t:.3f}')

## 3. Per-Model Metrics (PRD NFR2.1)

In [ ]:
def compute_metrics(y_true, y_pred, y_score, name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    fpr_val = fp / max(fp + tn, 1)
    return {
        'Model':     name,
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0),    4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0),        4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_score),                   4),
        'FPR':       round(fpr_val,                                           4),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }

m_if  = compute_metrics(y_test, if_pred,  if_score,  'Isolation Forest')
m_lof = compute_metrics(y_test, lof_pred, lof_score, 'LOF')

results = [m_if, m_lof]
df_results = pd.DataFrame(results)
display_cols = ['Model','Precision','Recall','F1','ROC-AUC','FPR','TP','TN','FP','FN']
print(df_results[display_cols].to_string(index=False))

## 4. Confusion Matrices

In [ ]:
def plot_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Normal','Anomaly'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Normal','Anomaly'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i,j], ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black',
                    fontsize=14, fontweight='bold')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_cm(y_test, if_pred,  'Isolation Forest', axes[0])
plot_cm(y_test, lof_pred, 'LOF',              axes[1])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices.png'), dpi=120)
plt.show()

## 5. ROC Curves (PRD §7.4.1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, score, color in [
    ('Isolation Forest', if_score,  'steelblue'),
    ('LOF',              lof_score, 'darkorange'),
]:
    fpr_a, tpr_a, _ = roc_curve(y_test, score)
    roc_val = roc_auc_score(y_test, score)
    ax.plot(fpr_a, tpr_a, lw=2, color=color, label=f'{name} (AUC={roc_val:.3f})')

ax.plot([0,1],[0,1],'k--',lw=1)
ax.axvline(PRD_FPR_TARGET, color='red', linestyle=':', lw=2, label=f'FPR target={PRD_FPR_TARGET}')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Tabular Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'), dpi=120)
plt.show()

## 6. Precision-Recall Curves (critical for 95/5 imbalance)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, score, color in [
    ('Isolation Forest', if_score,  'steelblue'),
    ('LOF',              lof_score, 'darkorange'),
]:
    prec_a, rec_a, _ = precision_recall_curve(y_test, score)
    ap = auc(rec_a, prec_a)
    ax.plot(rec_a, prec_a, lw=2, color=color, label=f'{name} (AP={ap:.3f})')

ax.axhline(PRD_PRECISION_MIN, color='green',  linestyle=':', lw=2,
           label=f'Precision target={PRD_PRECISION_MIN}')
ax.axvline(PRD_RECALL_MIN,    color='orange', linestyle=':', lw=2,
           label=f'Recall target={PRD_RECALL_MIN}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — Tabular Models')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pr_curves.png'), dpi=120)
plt.show()

## 7. Model Comparison Table & Bar Chart

In [ ]:
compare_df = pd.DataFrame(results)[['Model','Precision','Recall','F1','ROC-AUC','FPR']]
targets = pd.DataFrame([{
    'Model': 'PRD Target',
    'Precision': f'≥{PRD_PRECISION_MIN}',
    'Recall':    f'≥{PRD_RECALL_MIN}',
    'F1':        '—',
    'ROC-AUC':   f'≥{PRD_AUC_MIN}',
    'FPR':       f'≤{PRD_FPR_TARGET}',
}])
print(pd.concat([compare_df, targets], ignore_index=True).to_string(index=False))

# Bar chart
models = [r['Model'] for r in results]
x = np.arange(len(models))
w = 0.2
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x-1.5*w, [r['Precision'] for r in results], w, label='Precision', color='steelblue')
ax.bar(x-0.5*w, [r['Recall']    for r in results], w, label='Recall',    color='darkorange')
ax.bar(x+0.5*w, [r['F1']        for r in results], w, label='F1',        color='green')
ax.bar(x+1.5*w, [r['ROC-AUC']   for r in results], w, label='ROC-AUC',   color='purple')
ax.axhline(PRD_PRECISION_MIN, color='steelblue',  linestyle='--', lw=1, alpha=0.7)
ax.axhline(PRD_RECALL_MIN,    color='darkorange', linestyle='--', lw=1, alpha=0.7)
ax.axhline(PRD_AUC_MIN,       color='purple',     linestyle='--', lw=1, alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Model Comparison — PRD Targets (dashed)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison_bar.png'), dpi=120)
plt.show()

## 8. Feature Importance via SHAP (Isolation Forest, PRD R7)

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(iforest.model)
    sample = X_test.sample(min(500, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(sample)
    mean_abs = np.abs(shap_values).mean(axis=0)
    importance = pd.Series(mean_abs, index=X_test.columns).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    importance.head(15).plot(kind='barh', ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title('Feature Importance (SHAP) — Isolation Forest')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'shap_feature_importance.png'), dpi=120)
    plt.show()
    print('Top 5 features:', importance.head().to_dict())
except ImportError:
    print('shap not installed. Run: pip install shap')

## 9. K-Fold Cross-Validation (Isolation Forest stability)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from data_loader import load_dataset
from preprocessing import encode_categoricals, fit_scaler, apply_scaler

df_full = load_dataset()
df_enc, _ = encode_categoricals(df_full)
y_full = (df_enc['label'] != 0).astype(int)
X_full = df_enc.drop('label', axis=1)
scaler_cv = fit_scaler(X_full[y_full == 0])
X_full_s  = apply_scaler(scaler_cv, X_full)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_records = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_full_s, y_full), 1):
    X_tr = X_full_s.iloc[tr_idx]; y_tr = y_full.iloc[tr_idx]
    X_te = X_full_s.iloc[te_idx]; y_te = y_full.iloc[te_idx]
    X_tr_n = X_tr[y_tr == 0]
    det = IsolationForestDetector().fit(X_tr_n)
    sc  = det.anomaly_score(X_te.values)
    fpr_a, tpr_a, thr_a = roc_curve(y_te, sc)
    valid = np.where(fpr_a <= PRD_FPR_TARGET)[0]
    idx = min(valid[-1] if len(valid) else 0, len(thr_a)-1)
    det.set_threshold(float(thr_a[idx]))
    yp = det.predict(X_te.values)
    fold_records.append({
        'Fold': fold,
        'Precision': round(precision_score(y_te, yp, zero_division=0), 3),
        'Recall':    round(recall_score(y_te, yp, zero_division=0),    3),
        'F1':        round(f1_score(y_te, yp, zero_division=0),        3),
        'AUC':       round(roc_auc_score(y_te, sc),                    3),
    })

cv_df = pd.DataFrame(fold_records)
print(cv_df.to_string(index=False))
print('\nMean ± Std:')
print(cv_df.drop('Fold', axis=1).agg(['mean','std']).round(3).to_string())

## 10. Error Analysis — FP & FN Breakdown

In [ ]:
for name, y_pred, y_score in [
    ('Isolation Forest', if_pred,  if_score),
    ('LOF',              lof_pred, lof_score),
]:
    df_err = X_test.copy()
    df_err['y_true'] = y_test.values
    df_err['y_pred'] = y_pred
    df_err['score']  = y_score

    fp = df_err[(df_err['y_true']==0) & (df_err['y_pred']==1)]
    fn = df_err[(df_err['y_true']==1) & (df_err['y_pred']==0)]
    tp = df_err[(df_err['y_true']==1) & (df_err['y_pred']==1)]

    print(f'\n── {name} ──')
    print(f'  False Positives : {len(fp):5d}  avg score={fp["score"].mean():.4f}')
    print(f'  False Negatives : {len(fn):5d}  avg score={fn["score"].mean():.4f}')
    print(f'  True Positives  : {len(tp):5d}  avg score={tp["score"].mean():.4f}')

    # Score distribution plot
    fig, ax = plt.subplots(figsize=(8, 3))
    normal_scores = df_err[df_err['y_true']==0]['score']
    anomaly_scores = df_err[df_err['y_true']==1]['score']
    ax.hist(normal_scores,  bins=50, alpha=0.6, label='Normal', color='steelblue')
    ax.hist(anomaly_scores, bins=50, alpha=0.6, label='Anomaly', color='red')
    det_thr = iforest.threshold_ if name == 'Isolation Forest' else lof.threshold_
    ax.axvline(det_thr, color='black', linestyle='--', lw=2, label=f'Threshold={det_thr:.3f}')
    ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Count')
    ax.set_title(f'Score Distribution — {name}')
    ax.legend(); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'score_dist_{name.replace(" ","_")}.png'), dpi=120)
    plt.show()

## 11. Export metrics.json (PRD §11.3)

In [ ]:
for m in results:
    m['meets_precision'] = m['Precision'] >= PRD_PRECISION_MIN
    m['meets_recall']    = m['Recall']    >= PRD_RECALL_MIN
    m['meets_auc']       = m['ROC-AUC']   >= PRD_AUC_MIN
    m['meets_fpr']       = m['FPR']       <= PRD_FPR_TARGET

output = {
    'prd_targets': {
        'precision_min': PRD_PRECISION_MIN,
        'recall_min':    PRD_RECALL_MIN,
        'auc_min':       PRD_AUC_MIN,
        'fpr_max':       PRD_FPR_TARGET,
    },
    'model_results': results,
}

metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'metrics.json saved → {metrics_path}')
print(json.dumps(output, indent=2))

## 12. PRD Compliance Summary

In [ ]:
print('=' * 65)
print('PHASE 3 — PRD COMPLIANCE SUMMARY')
print('=' * 65)
hdr = f'{"Model":<22} {"Prec":>6} {"Rec":>6} {"F1":>6} {"AUC":>6} {"FPR":>6}  PRD'
print(hdr)
print('-' * len(hdr))
for m in results:
    ticks = (
        'P' if m.get('meets_precision') else 'x',
        'R' if m.get('meets_recall')    else 'x',
        'A' if m.get('meets_auc')       else 'x',
        'F' if m.get('meets_fpr')       else 'x',
    )
    print(f'{m["Model"]:<22} {m["Precision"]:>6.3f} {m["Recall"]:>6.3f} '
          f'{m["F1"]:>6.3f} {m["ROC-AUC"]:>6.3f} {m["FPR"]:>6.3f}  '
          f'[{"" .join(ticks)}]')
print('-' * len(hdr))
print('P=Precision>=0.85  R=Recall>=0.80  A=AUC>=0.90  F=FPR<=0.15')
print()
print('Phase 3 complete. Next -> Phase 4: streaming_simulator.py')